In [4]:
import os
import re
import shutil
from pathlib import Path
from typing import Optional, List, Dict, Any

import pandas as pd

In [2]:
os.chdir('minfin SEBRA files')

In [2]:
CSV_COLUMNS = [
    "Ref",
    "Operations Code",
    "Operations Description",
    "Operations Amount (EUR)",
    "Organization Name",
    "Start Date",
    "End Date",
    "Organization ID",
]


def _is_org_header(s: Any) -> bool:
    """Organization header lines look like: '<name> ( 055******* )'."""
    if not isinstance(s, str):
        return False
    t = s.strip()
    if not t:
        return False
    # Exclude the very first sheet title row "ОБЩО ПЛАЩАНИЯ ЗА ДЕНЯ (в евро)"
    if "ОБЩО ПЛАЩАНИЯ ЗА ДЕНЯ" in t:
        return False
    return "(" in t and ")" in t


def _parse_org_name_and_id(org_line: str) -> (str, str):
    # Example: 'Национален ... ( 055******* )'
    name = org_line.split("(", 1)[0].strip()
    m = re.search(r"\((.*?)\)", org_line)
    org_id = m.group(1).strip() if m else ""
    return name, org_id


def _parse_period(period_cell: Any) -> (str, str):
    """
    Period cell looks like: 'Период: 05.01.2026 - 05.01.2026'
    Returns dates as 'dd.mm.yyyy' strings.
    """
    if not isinstance(period_cell, str):
        return "", ""
    m = re.search(r"(\d{2}\.\d{2}\.\d{4})\s*-\s*(\d{2}\.\d{2}\.\d{4})", period_cell)
    if not m:
        return "", ""
    return m.group(1), m.group(2)


def _is_code_row(val: Any) -> bool:
    """Codes look like '01 xxxx', '40 xxxx', etc."""
    if not isinstance(val, str):
        return False
    return re.match(r"^\s*\d{2}\s*xxxx\s*$", val.strip(), flags=re.IGNORECASE) is not None


def parse_sebra_xlsx(xlsx_path: str) -> pd.DataFrame:
    """
    Parse one daily SEBRA XLSX into a normalized dataframe matching the target CSV structure,
    except for 'Ref' which is assigned when writing/appending to the combined CSV.
    """
    df = pd.read_excel(xlsx_path, sheet_name=0)  # first sheet is the dated one
    if df.shape[1] < 4:
        raise ValueError(f"Unexpected format (need 4 columns): {xlsx_path}")

    c0 = df.iloc[:, 0]
    c1 = df.iloc[:, 1]
    c2 = df.iloc[:, 2]
    c3 = df.iloc[:, 3]

    rows: List[Dict[str, Any]] = []
    i = 0
    n = len(df)

    while i < n:
        cell0 = c0.iat[i]

        if _is_org_header(cell0):
            org_name, org_id = _parse_org_name_and_id(str(cell0))
            start_date, end_date = _parse_period(c2.iat[i])

            # Sometimes period might not be on the same line; look ahead a bit
            if not start_date:
                for j in range(i, min(i + 4, n)):
                    sd, ed = _parse_period(c2.iat[j])
                    if sd:
                        start_date, end_date = sd, ed
                        break

            # Advance to the table rows (skip "Код" header line etc.)
            i += 1
            # Find the "Код" header line, then start after it
            while i < n and not (isinstance(c0.iat[i], str) and str(c0.iat[i]).strip() == "Код"):
                # If we hit another org header unexpectedly, break out
                if _is_org_header(c0.iat[i]):
                    break
                i += 1
            if i < n and isinstance(c0.iat[i], str) and str(c0.iat[i]).strip() == "Код":
                i += 1  # first data row after header

            # Read code lines until "Общо" or next org header
            while i < n:
                v0 = c0.iat[i]

                if _is_org_header(v0):
                    # new org starts
                    i -= 1  # step back so outer loop sees it next
                    break

                if isinstance(v0, str) and v0.strip().startswith("Общо"):
                    break

                if _is_code_row(v0):
                    code = str(v0).strip()
                    desc = c1.iat[i]
                    amt = c3.iat[i]

                    # normalize description
                    desc_str = "" if pd.isna(desc) else str(desc).strip()

                    # normalize amount -> float
                    amt_num = pd.to_numeric(amt, errors="coerce")

                    # Only keep rows with a numeric amount (including 0)
                    if pd.notna(amt_num):
                        rows.append(
                            {
                                "Operations Code": code,
                                "Operations Description": desc_str,
                                "Operations Amount (EUR)": float(amt_num),
                                "Organization Name": org_name,
                                "Start Date": start_date,
                                "End Date": end_date,
                                "Organization ID": org_id,
                            }
                        )

                i += 1

        i += 1

    out = pd.DataFrame(rows, columns=[c for c in CSV_COLUMNS if c != "Ref"])
    return out


def process_sebra_folder(
    folder: str,
    output_csv: str,
    processed_subfolder: str = "processed",
    encoding: str = "utf-8-sig",
) -> str:
    """
    Process all SEBRA-*.xlsx files in 'folder', append to 'output_csv',
    and move successfully processed files to '<folder>/<processed_subfolder>/'.

    Returns: output_csv path
    """
    folder_path = Path(folder)
    processed_path = folder_path / processed_subfolder
    processed_path.mkdir(parents=True, exist_ok=True)

    # Determine starting Ref (continue if CSV exists)
    out_path = Path(output_csv)
    if out_path.exists():
        existing = pd.read_csv(out_path, encoding=encoding)
        max_ref = pd.to_numeric(existing["Ref"], errors="coerce").max()
        next_ref = int(max_ref) + 1 if pd.notna(max_ref) else 1
    else:
        next_ref = 1

    # Find files like SEBRA-05012026.xlsx
    xlsx_files = sorted(folder_path.glob("SEBRA-*.xlsx"))

    appended_any = False

    for xlsx in xlsx_files:
        # Skip anything already inside processed folder
        if processed_path in xlsx.parents:
            continue

        try:
            daily = parse_sebra_xlsx(str(xlsx))
            if daily.empty:
                # Still consider it "processed" if format was okay but nothing to add
                shutil.move(str(xlsx), str(processed_path / xlsx.name))
                continue

            # Assign Ref
            daily.insert(0, "Ref", range(next_ref, next_ref + len(daily)))
            next_ref += len(daily)

            # Append / write
            if out_path.exists():
                daily.to_csv(out_path, mode="a", header=False, index=False, encoding=encoding)
            else:
                # Ensure column order exactly matches target
                daily = daily[CSV_COLUMNS]
                daily.to_csv(out_path, mode="w", header=True, index=False, encoding=encoding)

            appended_any = True

            # Move to processed after successful append
            shutil.move(str(xlsx), str(processed_path / xlsx.name))

        except Exception as e:
            # Leave the file in place so you can inspect/retry; raise for visibility
            raise RuntimeError(f"Failed processing {xlsx.name}: {e}") from e

    # If CSV existed but we only appended, ensure columns match (optional safety)
    if out_path.exists() and appended_any:
        # no-op; kept for clarity
        pass

        # Append Category column from categories.csv
    categories_path = folder_path / "categories.csv"

    if categories_path.exists() and out_path.exists():
        output_df = pd.read_csv(out_path, encoding=encoding)
        categories_df = pd.read_csv(categories_path, encoding=encoding)

        # Keep only needed columns
        categories_df = categories_df[["Organization ID", "Category"]]

        # Clean join keys
        output_df["Organization ID"] = output_df["Organization ID"].astype(str).str.strip()
        categories_df["Organization ID"] = categories_df["Organization ID"].astype(str).str.strip()

        if "Category" in output_df.columns:
            output_df = output_df.drop(columns=["Category"])

        # Merge category
        output_df = output_df.merge(
            categories_df,
            on="Organization ID",
            how="left"
        )

        # Save updated file
        output_df.to_csv(out_path, index=False, encoding=encoding)

    return str(out_path)

In [ ]:
import hashlib
import re


def _parse_period(period_str):
    """Extract start and end dates from period string like 'Период: 19.05.2023 - 19.05.2023'."""
    if not isinstance(period_str, str):
        return None, None
    m = re.search(r'(\d{2}\.\d{2}\.\d{4})\s*-\s*(\d{2}\.\d{2}\.\d{4})', period_str)
    if m:
        return m.group(1), m.group(2)
    return None, None


def _parse_org_header(header_str):
    """Parse organization name and ID from string like 'Name ( ID )'."""
    if not isinstance(header_str, str):
        return '', None
    name = header_str.split('(', 1)[0].strip()
    m = re.search(r'\((.*?)\)', header_str)
    org_id = m.group(1).strip() if m else None
    return name, org_id


def _generate_code_from_text(text, length=7):
    """Generate a code by uppercasing text, computing MD5, and taking first N chars."""
    return hashlib.md5(text.upper().encode()).hexdigest()[:length]


def _is_operation_code(val):
    """Check if string matches operation code pattern like '01 xxxx'."""
    if not isinstance(val, str):
        return False
    return bool(re.match(r'^\s*\d{2}\s*xxxx\s*$', val.strip(), flags=re.IGNORECASE))


def _is_descriptive_row(row):
    """Check if row has value only in first column (descriptive/skippable)."""
    if pd.isna(row[0]) or not str(row[0]).strip():
        return False
    return all(pd.isna(v) or not str(v).strip() for v in row[1:])


def parse_sebra_payments_xlsx(xlsx_path: str) -> pd.DataFrame:
    """
    Parses a xlsx file with SEBRA payments and returns two DataFrames:
    * The first one contains the rows extracted from the summary info at
      the beginning of the file:
      Its columns are:
      * start_date
      * end_date
      * operation_code
      * operation_description
      * currency - could be BGN or EUR
      * amount
    * The second one contains the data for payments made by each organization
      * start_date
      * end_date
      * operation_code
      * operation_description
      * currency - could be BGN or EUR
      * amount
      * organization_id
      * organization_name

    Parsing steps:
    * after the xlsx file is loaded into a pd.DataFrame, all rows where all columns are NaN are dropped
    * the DataFrame should contain 4 columns, if not the process is interrupted with RuntimeException
    * the DataFrame is a sequence of multiple sections
    * at the very top there is a summary section which contains summarized amounts
      by operations codes
      * this section starts with "ОБЩО ПЛАЩАНИЯ ЗА ДЕНЯ (в евро)" or
        "ОБЩО ПЛАЩАНИЯ ЗА ДЕНЯ (в лева)" in the first column
        * the value within the brackets defines the currency
      * on the same row in the third column there's the period which looks like
        "Период: 19.05.2023 - 19.05.2023"
        * the two dates are in DD.MM.YYYY format
      * a summary row within the section contains
        * in the first column an operation code in the form "01 xxxx"
          * there are operations without dedicated code and in this case
            this column contains "    xxxx"
          * for operations without dedicated operation code one will be
            calculated by uppercasing the operation name and calculating its
            md5 sum. The final value will be first 7 chars from the md5 sum.
        * in the second column there's the operation code
        * the third column is empty
        * the forth column contains the amount
      * this section may contain separating lines which provide additional
        descriptive row with text only in the first column, these lines will be skipped
      * the section completes with a row which contains "Общо: " in the first row
        and total sum in the forth column
    * The summary section might be followed by a descriptive row which has value
      only in the first column, this line will be skipped
    * After that there will be one or more sections payment operations, one
      section for each organization
      * Such a section starts with a row in which the first column looks like
        "Народно събрание ( 001******* )" where the text before the opening
        bracket is the organization name, while the text within the brackets
        is the organization id
        * In case there is no brackets section, then an organization ID will
          be calculated by uppercasing the contents of the first column and getting
          the first 10 chars of its md5 sum
      * In the third column there will be the period column which is in the same
        format as the one above Период: 19.05.2023 - 19.05.2023
      * After that there could be a row contain column headers like "Код", "Описание"
        and "Сума" - this row is skipped
      * Then follow the operations rows; each row has
        * the operation code in the first column in the form "01 xxxx"
          if the code does not match the pattern, then the same approach for building
          a code as the one described above will be applied
        * the operation description in the second column
        * the amount in the forth column
      * A section for an organization completes with a row which contains
        "Общо: " in the first column and the total sum in the forth column
    """
    df = pd.read_excel(xlsx_path, sheet_name=0)
    df = df.dropna(how='all')

    if df.shape[1] != 4:
        raise RuntimeError(f'Expected 4 columns, got {df.shape[1]}: {xlsx_path}')

    c0 = df.iloc[:, 0]
    c1 = df.iloc[:, 1]
    c2 = df.iloc[:, 2]
    c3 = df.iloc[:, 3]

    summary_rows = []
    payment_rows = []
    currency = None
    summary_start_date = None
    summary_end_date = None

    i = 0
    n = len(df)

    # Parse summary section
    while i < n:
        row0_val = c0.iat[i]
        row = (row0_val, c1.iat[i], c2.iat[i], c3.iat[i])

        # Check for summary section start
        if isinstance(row0_val, str) and 'ОБЩО ПЛАЩАНИЯ ЗА ДЕНЯ' in row0_val:
            m = re.search(r'\((в\s*[а-яА-Я]+)\)', row0_val)
            if m:
                currency_str = m.group(1).strip().lower()
                currency = 'EUR' if 'евро' in currency_str else 'BGN'

            start, end = _parse_period(c2.iat[i])
            if start and end:
                summary_start_date, summary_end_date = start, end

            i += 1
            continue

        # Skip descriptive rows
        if _is_descriptive_row(row):
            i += 1
            continue

        # Check for summary section end
        if isinstance(row0_val, str) and 'Общо:' in row0_val:
            i += 1
            break

        # Parse summary operation rows
        if summary_start_date and currency:
            code = str(row0_val).strip() if pd.notna(row0_val) else ''
            desc = str(c1.iat[i]).strip() if pd.notna(c1.iat[i]) else ''
            amt = c3.iat[i]

            if not _is_operation_code(code):
                if code and code.strip() != '':
                    code = _generate_code_from_text(desc if desc else code, 7)
                else:
                    code = ''

            try:
                amt_val = float(amt) if pd.notna(amt) else 0.0
            except (ValueError, TypeError):
                i += 1
                continue

            summary_rows.append({
                'start_date': summary_start_date,
                'end_date': summary_end_date,
                'operation_code': code,
                'operation_description': desc,
                'currency': currency,
                'amount': amt_val
            })

            i += 1
            continue

        if pd.isna(row0_val) or not str(row0_val).strip():
            i += 1
            continue

        if isinstance(row0_val, str) and '(' in row0_val and ')' in row0_val:
            break

        i += 1

    # Parse organization sections
    while i < n:
        row0_val = c0.iat[i]
        row = (row0_val, c1.iat[i], c2.iat[i], c3.iat[i])

        if _is_descriptive_row(row):
            i += 1
            continue

        if isinstance(row0_val, str) and '(' in row0_val and ')' in row0_val:
            org_name, org_id = _parse_org_header(row0_val)
            if org_id is None:
                org_id = _generate_code_from_text(org_name, 10)

            org_start, org_end = _parse_period(c2.iat[i])

            i += 1

            if i < n:
                next_row0 = str(c0.iat[i]).strip() if pd.notna(c0.iat[i]) else ''
                if 'Код' in next_row0:
                    i += 1

            while i < n:
                curr_row0 = c0.iat[i]
                curr_row = (curr_row0, c1.iat[i], c2.iat[i], c3.iat[i])

                if isinstance(curr_row0, str) and 'Общо:' in curr_row0:
                    i += 1
                    break

                if _is_descriptive_row(curr_row):
                    i += 1
                    continue

                if pd.isna(curr_row0) or not str(curr_row0).strip():
                    i += 1
                    continue

                code = str(curr_row0).strip() if pd.notna(curr_row0) else ''
                desc = str(c1.iat[i]).strip() if pd.notna(c1.iat[i]) else ''
                amt = c3.iat[i]

                if not _is_operation_code(code):
                    code = _generate_code_from_text(desc if desc else code, 7) if code.strip() else ''

                try:
                    amt_val = float(amt) if pd.notna(amt) else 0.0
                except (ValueError, TypeError):
                    i += 1
                    continue

                payment_rows.append({
                    'start_date': org_start,
                    'end_date': org_end,
                    'operation_code': code,
                    'operation_description': desc,
                    'currency': currency or 'EUR',
                    'amount': amt_val,
                    'organization_id': org_id,
                    'organization_name': org_name
                })

                i += 1

            continue

        i += 1

    summary_df = pd.DataFrame(summary_rows)
    payments_df = pd.DataFrame(payment_rows)

    return summary_df, payments_df

In [6]:
sdf, odf = parse_sebra_payments_xlsx('./Data/SEBRA-09042026.xlsx')

In [10]:
sdf


,start_date,end_date,operation_code,operation_description,currency,amount
0,09.04.2026,09.04.2026,01 xxxx,"Заплати, възнаграждения и други плащания за пе...",EUR,6.364717e+06
1,09.04.2026,09.04.2026,02 xxxx,Удържани данъци и осигурителни вноски от запла...,EUR,1.625345e+04
2,09.04.2026,09.04.2026,03 xxxx,Плащания за други удръжки от възнаграждения за...,EUR,2.236844e+04
3,09.04.2026,09.04.2026,05 xxxx,Осигурителни вноски за сметка на осигурителя,EUR,1.225326e+04
4,09.04.2026,09.04.2026,10 xxxx,Издръжка,EUR,1.459460e+07
5,09.04.2026,09.04.2026,18 xxxx,Други разходи,EUR,6.957840e+05
6,09.04.2026,09.04.2026,20 xxxx,Разходи за лихви,EUR,1.513000e+01
7,09.04.2026,09.04.2026,30 xxxx,Текущи субсидии за предприятия,EUR,4.482515e+06
8,09.04.2026,09.04.2026,40 xxxx,"Стипендии, пенсии, помощи и текущи трансфери з...",EUR,1.114603e+08
9,09.04.2026,09.04.2026,50 xxxx,"Плащания за дълготрайни активи, основен ремонт...",EUR,6.291945e+06


In [8]:
odf

,start_date,end_date,operation_code,operation_description,currency,amount,organization_id,organization_name
0,09.04.2026,09.04.2026,01 xxxx,"Заплати, възнаграждения и други плащания за пе...",EUR,17534.74,001*******,Народно събрание
1,09.04.2026,09.04.2026,10 xxxx,Издръжка,EUR,492960.94,001*******,Народно събрание
2,09.04.2026,09.04.2026,90 xxxx,Възстановени приходи,EUR,1436.55,001*******,Народно събрание
3,09.04.2026,09.04.2026,01 xxxx,"Заплати, възнаграждения и други плащания за пе...",EUR,55594.53,003*******,Министерски съвет
4,09.04.2026,09.04.2026,10 xxxx,Издръжка,EUR,83802.68,003*******,Министерски съвет
...,...,...,...,...,...,...,...,...
217,09.04.2026,09.04.2026,89 xxxx,Друго финансиране,EUR,2535183.76,983*******,Национален фонд - Механизъм за възстановяване ...
218,09.04.2026,09.04.2026,10 xxxx,Издръжка,EUR,3095.90,987*******,Национален фонд - Средства от Европейския съюз
219,09.04.2026,09.04.2026,30 xxxx,Текущи субсидии за предприятия,EUR,704998.16,987*******,Национален фонд - Средства от Европейския съюз
220,09.04.2026,09.04.2026,60 xxxx,Трансфери за бюджетни и извънбюджетни сметки,EUR,1971681.79,987*******,Национален фонд - Средства от Европейския съюз


In [ ]:
parse_sebra_xlsx('./Data/SEBRA-09042026.xlsx')


# output = process_sebra_folder(
#     folder="minfin SEBRA files",
#     output_csv="SEBRA2026EUR.csv",
#     encoding="utf-8-sig",
# )

,Operations Code,Operations Description,Operations Amount (EUR),Organization Name,Start Date,End Date,Organization ID
0,01 xxxx,"Заплати, възнаграждения и други плащания за пе...",17534.74,Народно събрание,09.04.2026,09.04.2026,001*******
1,10 xxxx,Издръжка,492960.94,Народно събрание,09.04.2026,09.04.2026,001*******
2,90 xxxx,Възстановени приходи,1436.55,Народно събрание,09.04.2026,09.04.2026,001*******
3,01 xxxx,"Заплати, възнаграждения и други плащания за пе...",55594.53,Министерски съвет,09.04.2026,09.04.2026,003*******
4,10 xxxx,Издръжка,83802.68,Министерски съвет,09.04.2026,09.04.2026,003*******
...,...,...,...,...,...,...,...
217,89 xxxx,Друго финансиране,2535183.76,Национален фонд - Механизъм за възстановяване ...,09.04.2026,09.04.2026,983*******
218,10 xxxx,Издръжка,3095.90,Национален фонд - Средства от Европейския съюз,09.04.2026,09.04.2026,987*******
219,30 xxxx,Текущи субсидии за предприятия,704998.16,Национален фонд - Средства от Европейския съюз,09.04.2026,09.04.2026,987*******
220,60 xxxx,Трансфери за бюджетни и извънбюджетни сметки,1971681.79,Национален фонд - Средства от Европейския съюз,09.04.2026,09.04.2026,987*******


In [6]:
raw_data = pd.read_excel('./Data/SEBRA-09042026.xlsx', sheet_name=0)
raw_data

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3
0,ОБЩО ПЛАЩАНИЯ ЗА ДЕНЯ (в евро),NaN,Период: 09.04.2026 - 09.04.2026,NaN
1,Код,Описание,NaN,Сума
2,NaN,NaN,NaN,NaN
3,"I. ПЛАЩАНИЯ ОТ БЮДЖЕТА, ИЗВЪРШЕНИ ЧРЕЗ СЕБРА, ...",NaN,NaN,NaN
4,NaN,NaN,NaN,NaN
...,...,...,...,...
584,NaN,NaN,NaN,NaN
585,Други трансфери за общини ( 488******* ),NaN,Период: 09.04.2026 - 09.04.2026,NaN
586,NaN,Описание,NaN,Сума
587,NaN,Трансфери за други целеви разходи за общини,NaN,682181.8


In [16]:
stage1 = raw_data.dropna(how='all').reset_index(drop=True)
stage1[0:50]

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3
0,ОБЩО ПЛАЩАНИЯ ЗА ДЕНЯ (в евро),NaN,Период: 09.04.2026 - 09.04.2026,NaN
1,Код,Описание,NaN,Сума
2,"I. ПЛАЩАНИЯ ОТ БЮДЖЕТА, ИЗВЪРШЕНИ ЧРЕЗ СЕБРА, ...",NaN,NaN,NaN
3,01 xxxx,"Заплати, възнаграждения и други плащания за пе...",NaN,6364717.34
4,02 xxxx,Удържани данъци и осигурителни вноски от запла...,NaN,16253.45
5,03 xxxx,Плащания за други удръжки от възнаграждения за...,NaN,22368.44
6,05 xxxx,Осигурителни вноски за сметка на осигурителя,NaN,12253.26
7,10 xxxx,Издръжка,NaN,14594595.12
8,18 xxxx,Други разходи,NaN,695783.96
9,20 xxxx,Разходи за лихви,NaN,15.13
